# 01 — Data audit

This notebook documents the authoritative MonReader image dataset. It reads the committed immutable registration and manifest; ingestion, hashing, and validation logic remain in reusable source modules. The raw `images.zip` archive is intentionally not stored in Git.

In [ ]:
from pathlib import Path
import sys
import json
import yaml

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data import load_manifest, sha256_file, summarize_manifest
from src.visualization import plot_image_grid

DATASET_DIR = ROOT / "manifests" / "datasets"
SAMPLE_DIR = ROOT / "docs" / "sample_images"
MANIFEST_PATH = DATASET_DIR / "DATA-001.csv"

## Authoritative dataset identity

The project accepts one source archive only. The committed registration records the archive filename, byte size, SHA-256 hash, labels, supplied split membership, and canonical manifest hash.

In [ ]:
registration = yaml.safe_load((DATASET_DIR / "DATA-001.yaml").read_text(encoding="utf-8"))
summary = json.loads((DATASET_DIR / "DATA-001_summary.json").read_text(encoding="utf-8"))
manifest_rows = load_manifest(MANIFEST_PATH)

observed_manifest_sha = sha256_file(MANIFEST_PATH)
assert observed_manifest_sha == registration["manifest"]["sha256"]
assert len(manifest_rows) == registration["manifest"]["row_count"] == summary["image_count"]

print(json.dumps(registration["source_archive"], indent=2))
print(f"manifest rows: {len(manifest_rows):,}")
print(f"manifest sha256: {observed_manifest_sha}")

## Composition and native image geometry

The supplied `training` / `testing` membership is preserved as the **Original** benchmark. Later notebooks construct alternative evaluation regimes without changing the supplied labels.

In [ ]:
manifest_summary = summarize_manifest(MANIFEST_PATH)
print(json.dumps(manifest_summary, indent=2, sort_keys=True))
print("\nSplit × label counts:")
print(json.dumps(summary["split_label_counts"], indent=2, sort_keys=True))
print("\nNative dimensions:")
print(json.dumps(summary["dimension_counts"], indent=2, sort_keys=True))

## Video and frame naming

Archive members follow `images/{training|testing}/{flip|notflip}/{raw_video_id}_{FrameNumber}.jpg`. The four-digit numeric namespace is reused between class folders, so the immutable manifest stores both the raw numeric ID and a class-namespaced `video_id = {label}/{raw_video_id}`. This avoids silently merging ambiguous sequences. Broader source/environment grouping is reviewed separately during split design. Temporal ordering uses ascending `FrameNumber` within the resolved video/source group.

In [ ]:
overlap = summary["supplied_split_video_counts"]
print(f"raw numeric video IDs: {summary['raw_video_id_count']}")
print(f"canonical video groups: {summary['video_count']}")
print(json.dumps(overlap, indent=2, sort_keys=True))
print(
    f"\n{overlap['cross_split']} of {summary['video_count']} canonical video groups "
    "occur in both supplied partitions; the supplied split is therefore not an unseen-video test."
)

## Exact decoded-pixel duplicate check

Exact duplicates are defined by identical decoded RGB pixel arrays, not filenames or compressed JPEG bytes. Near-duplicate analysis is not part of the primary study.

In [ ]:
duplicate_summary = {
    "duplicate_groups": summary["exact_pixel_duplicate_group_count"],
    "images_in_duplicate_groups": summary["exact_pixel_duplicate_image_count"],
    "cross_label_groups": summary["cross_label_exact_duplicate_group_count"],
    "cross_split_groups": summary["cross_split_exact_duplicate_group_count"],
}
print(json.dumps(duplicate_summary, indent=2))

## Training-only RGB statistics

These statistics use only the supplied training population. Later split-specific preprocessing must recompute training-only statistics whenever the training population changes.

In [ ]:
print("RGB mean:", summary["training_rgb_mean"])
print("RGB std: ", summary["training_rgb_std"])
print("training pixels:", f"{summary['training_rgb_pixel_count']:,}")

## Representative images

These eight images are a deterministic, stratified documentation and smoke-test sample only. They are not a modeling subset.

In [ ]:
sample_manifest = load_manifest(SAMPLE_DIR / "sample_manifest.csv")
sample_paths = [
    SAMPLE_DIR / f"{row['supplied_split']}_{row['label']}_{row['raw_video_id']}_{int(row['frame_number']):09d}.jpg"
    for row in sample_manifest
]
sample_titles = [
    f"{row['supplied_split']} | {row['label']} | {row['video_id']} | frame {row['frame_number']}"
    for row in sample_manifest
]
fig = plot_image_grid(sample_paths, titles=sample_titles, columns=4, figsize=(12, 10))
fig

## Dataset lineage starting point

The original RGB dataset is immutable. Later grayscale, duplicate-filtered, resolution, normalization, augmentation, and hand-removal variants must be registered as explicit descendants rather than overwriting it.

In [ ]:
lineage = yaml.safe_load((ROOT / "manifests" / "dataset_lineage.yaml").read_text(encoding="utf-8"))
print(yaml.safe_dump(lineage, sort_keys=False))

## Data-review conclusion

The archive identity is fixed, every sample decoded successfully during manifest generation, labels and frame metadata were validated, exact decoded-pixel hashes are available for every image, and the original RGB dataset is registered with immutable lineage. Split construction and broader source/environment grouping are handled next; no model result is inferred here.